# Red de Transporte Público de Tenerife (TITSA)
## Modelado de Complejidad Topológica, Centralidad y Resiliencia ante Disrupciones

Este notebook analiza la topología insular de transporte público construida a partir de las 181 líneas de TITSA, identificando cuellos de botella estructurales, agrupamientos comunitarios (zonas funcionales de movilidad) y la curva de degradación de servicio ante fallos aleatorios o ataques dirigidos a corredores troncales.

In [ ]:
import os
import sys
import json
import networkx as nx
import matplotlib.pyplot as plt

base_dir = os.path.dirname(os.getcwd())
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)

from src.graph.builder import load_snapshot
from src.graph.analytics import (
    compute_centrality_metrics,
    detect_communities,
    simulate_disruption,
    collapse_to_simple_digraph
)

snapshot_file = os.path.join(base_dir, "data", "processed", "graph_snapshot.json")
graph = load_snapshot(snapshot_file)
print(f"Nodos cargados: {graph.number_of_nodes()}")
print(f"Aristas dirigidas: {graph.number_of_edges()}")

### 1. Estadísticas Globales de la Red MultiDiGraph

In [ ]:
simple_g = collapse_to_simple_digraph(graph)
density = nx.density(simple_g)
in_degrees = [d for _, d in simple_g.in_degree()]
out_degrees = [d for _, d in simple_g.out_degree()]

print(f"Densidad del grafo simple: {density:.4f}")
print(f"Grado entrante promedio: {sum(in_degrees) / len(in_degrees):.2f}")
print(f"Grado saliente promedio: {sum(out_degrees) / len(out_degrees):.2f}")

### 2. Centralidad de Intermediación (Betweenness Centrality)
Identifica los nodos puente críticos por donde transcurre la mayor proporción de itinerarios mínimos entre el Norte, Sur, Área Metropolitana e interior montañoso.

In [ ]:
centrality_data = compute_centrality_metrics(graph, top_k=10)
names = [item["name"] for item in reversed(centrality_data)]
bc_scores = [item["betweenness_centrality"] for item in reversed(centrality_data)]

plt.figure(figsize=(10, 6))
bars = plt.barh(names, bc_scores, color="royalblue", edgecolor="black")
plt.xlabel("Betweenness Centrality")
plt.title("Top 10 Hubs Críticos por Centralidad de Intermediación en Tenerife")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 3. Comunidades de Movilidad (Algoritmo Louvain)
Agrupamiento modular de la red basado en la frecuencia acumulada de paso entre paradas e intercambiadores.

In [ ]:
communities = detect_communities(graph)
for comm in communities[:6]:
    print(f"[{comm['label']}] - Tamaño: {comm['size']} nodos")
    sample_nodes = [graph.nodes[n].get("name", n) for n in comm["nodes"][:5]]
    print("  Paradas principales:", ", ".join(sample_nodes))
    print("-" * 50)

### 4. Simulación de Percolación y Vulnerabilidad ante Cierres
Evaluación del impacto en la red al suspender líneas troncales (010 Santa Cruz - Sur TFS, 015 Santa Cruz - La Laguna, 110 Santa Cruz - Costa Adeje, 100 Santa Cruz - Puerto de la Cruz).

In [ ]:
scenarios = [
    ("Base (0 cierres)", [], []),
    ("Corte L015 (SC - La Laguna Directa)", ["015"], []),
    ("Corte Troncal TF-5 (L015 + L100 + L102)", ["015", "100", "102"], []),
    ("Corte Troncal TF-1 (L010 + L110 + L111)", ["010", "110", "111"], []),
    ("Aislamiento Hub Santa Cruz", [], ["INTERCAMBIADOR_SANTA_CRUZ"])
]

results = []
for label, lines, nodes in scenarios:
    sim = simulate_disruption(graph, removed_lines=lines, closed_nodes=nodes)
    results.append({
        "scenario": label,
        "disconnected_pct": sim["disconnected_pairs_pct"],
        "avg_delay_pct": sim["avg_travel_time_increase_pct"],
        "isolated_count": sim["isolated_nodes_count"]
    })

for r in results:
    print(f"{r['scenario']}:")
    print(f"  Pares desconectados: {r['disconnected_pct']}%")
    print(f"  Incremento medio en tiempo de viaje: +{r['avg_delay_pct']}%")
    print(f"  Nodos totalmente aislados: {r['isolated_count']}")
    print()"